In [ ]:
import json, os, subprocess, shutil, time, base64, requests
from pathlib import Path
cfg = json.load(open("config.json"))
CAPTIONS_LIST = cfg["captions_list"]
DURATION = cfg["duration"]
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
GITHUB_REPO = "keizdespania/LofiProducer-backup"


In [ ]:
print("Extrayendo runtime...", flush=True)
subprocess.run(["tar", "-xzf", "/kaggle/input/acestep-runtime/acestep-runtime.tgz", "-C", "/kaggle/working/"], check=True)
REPO = "/kaggle/working/acestep.cpp"
os.chdir(REPO)


In [ ]:
request = {
    "lm_model": "acestep-5Hz-lm-1.7B-Q8_0.gguf",
    "synth_model": "acestep-v15-xl-turbo-Q4_K_M.gguf",
    "caption": "",
    "task_type": "text2music",
    "lyrics": "",
    "instrumental": True,
    "vocal_language": "unknown",
    "duration": DURATION,
    "bpm": 90,
    "keyscale": "",
    "timesignature": "4/4",
    "inference_steps": 6,
    "guidance_scale": 1.0,
    "shift": 3.0,
    "lm_temperature": 0.85,
    "lm_cfg_scale": 2.0,
    "lm_top_p": 0.9,
    "seed": -1,
    "output_format": "mp3",
    "mp3_bitrate": 128
}
T_START = time.time()
for i, caption in enumerate(CAPTIONS_LIST):
    if time.time() - T_START > 55*60: break
    request["caption"] = caption
    open("/kaggle/working/request.json", "w").write(json.dumps(request))
    print(f"Generando {i+1}/{len(CAPTIONS_LIST)}...", flush=True)
    subprocess.run(["./build/ace-lm", "--models", "models", "--request", "/kaggle/working/request.json"], check=True)
    subprocess.run(["./build/ace-synth", "--models", "models", "--request", "/kaggle/working/request0.json"], check=True)
    candidates = list(Path("/kaggle/working").glob("request00*.mp3"))
    if candidates:
        mp3 = candidates[0]
        ts = int(time.time())
        dest_name = f"musica_{ts}_{i}.mp3"
        dst = Path("/kaggle/working") / dest_name
        shutil.copy2(mp3, dst)
        if GITHUB_TOKEN:
            content = base64.b64encode(dst.read_bytes()).decode()
            url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/output/musica/{dest_name}"
            headers = {"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3+json"}
            r = requests.get(url, headers=headers)
            sha = r.json().get("sha") if r.status_code == 200 else None
            payload = {"message": f"Add music {dest_name}", "content": content, "branch": "main"}
            if sha: payload["sha"] = sha
            requests.put(url, headers=headers, json=payload)
        print(f"OK: {dest_name}", flush=True)
